# Phase 5 — Serving, Deployment, and Cross-Repo Lineage

End-to-end demo:

1. Train a toy LightGBM alpha on Alpha158 with a `PreprocessingSpec` attached.
2. Log + register it with MLflow via `register_and_serve`.
3. Deploy it via Ray Serve; check the endpoint and shut it down.
4. Package it for TorchServe (no live server needed to demo packaging).
5. Emit lineage events to `agentic_assistants`.
6. Inspect the Prometheus metric surface.

Deps: `pip install "agentic-quant-platform[ml,ml-torch,serving]"` and a reachable MLflow tracking server (`AQP_MLFLOW_TRACKING_URI`).

## 1. Train + attach a PreprocessingSpec

In [ ]:
import pickle
from pathlib import Path
from aqp.ml.processors import (
    CSZScoreNorm, DropnaLabel, Fillna, PreprocessingSpec,
)

# Toy stand-in for a real Model — the real training lives in phase 2.
class ToyAlpha:
    def __init__(self):
        self.preprocessing_spec = None

    def predict(self, df):
        return [0.01] * len(df)

    def with_preprocessing(self, spec):
        self.preprocessing_spec = spec
        return self

processors = [Fillna(), CSZScoreNorm(), DropnaLabel()]
spec = PreprocessingSpec.from_processors(
    processors=processors,
    feature_columns=[f'f{i}' for i in range(10)],
    label_column='label_5d',
    metadata={'fit_window': '2020..2024', 'dataset_hash': 'demo'},
)

alpha = ToyAlpha().with_preprocessing(spec)
artifact_path = Path('/tmp/aqp-demo/alpha.pkl')
artifact_path.parent.mkdir(parents=True, exist_ok=True)
with open(artifact_path, 'wb') as fh:
    pickle.dump(alpha, fh)
print('saved to', artifact_path)

## 2. Register + serve via MLflow Models

In [ ]:
from aqp.mlops.mlflow_client import register_and_serve

# Replace the path with a real MLflow artifact URI in production.
info = register_and_serve(
    model_uri=str(artifact_path),
    name='aqp-demo-alpha',
    backend='mlflow',   # try 'ray' or 'torchserve' as alternates
    stage='Staging',
    host='127.0.0.1',
    port=5001,
)
info

## 3. Ray Serve deployment

Requires `ray[serve]`. Runs a local head if none is reachable.

In [ ]:
import contextlib
from aqp.mlops.serving.base import resolve_model
from aqp.mlops.serving.ray_serve import RayServeDeployment

with contextlib.suppress(Exception):
    deployer = RayServeDeployment(num_replicas=1)
    info = deployer.deploy(resolve_model(str(artifact_path)))
    print('ray serve endpoint:', info.endpoint_url)
    deployer.stop(info)

## 4. Package for TorchServe

This only packages the .mar; registering requires a running TorchServe.

In [ ]:
import contextlib
from pathlib import Path
from aqp.mlops.serving.torchserve import TorchServeDeployment

with contextlib.suppress(Exception):
    ts = TorchServeDeployment(model_store=Path('/tmp/aqp-demo/torchserve'))
    mar = ts.package_mar(
        model=resolve_model(str(artifact_path)),
        model_name='aqp-demo-alpha',
        version='1.0',
    )
    print('packaged', mar)

## 5. Emit lineage events

In [ ]:
from aqp.mlops import (
    emit_dataset, emit_run, emit_model, emit_serve_deployment,
)

emit_dataset('demo-hash', n_rows=1_000, n_symbols=10)
emit_run('demo-run', kind='alpha_training', dataset_hash='demo-hash', model_class='ToyAlpha')
emit_model('aqp-demo-alpha', version='1', run_id='demo-run')
emit_serve_deployment(
    endpoint_url='http://localhost:5001',
    backend='mlflow-serve',
    model_uri='models:/aqp-demo-alpha/Staging',
)

## 6. Prometheus metrics

In [ ]:
from aqp.mlops import (
    TRAIN_DURATION, BACKTEST_SHARPE, PAPER_PNL, time_histogram,
)

with time_histogram(TRAIN_DURATION, 'LightGBMAlpha'):
    # ... actual training work here ...
    pass

BACKTEST_SHARPE.labels('MomentumAlpha').set(1.74)
PAPER_PNL.labels('MomentumAlpha', 'alpaca-paper-1').set(123.45)

print('metrics ready; scraped by Prometheus from :9300')